# autoresearch-qc — Experiment Analysis

Reads `results.tsv` and generates plots to visualize agent experiment progress.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from pathlib import Path

plt.style.use("seaborn-v0_8-whitegrid")

CHEMICAL_ACCURACY_HA = 0.0016
RESULTS_FILE = Path("results.tsv")

df = pd.read_csv(RESULTS_FILE, sep="\t")
df["experiment_num"] = range(1, len(df) + 1)
print(f"Loaded {len(df)} experiments")
df.head()

## 1. Progress Plot

Energy error vs experiment number (log scale). Green = improvement kept, red = reverted.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# Color by whether the experiment improved (was kept)
colors = []
best_so_far = float("inf")
for _, row in df.iterrows():
    if row["energy_error"] < best_so_far:
        colors.append("#2ecc71")  # green — kept
        best_so_far = row["energy_error"]
    else:
        colors.append("#e74c3c")  # red — reverted

ax.scatter(df["experiment_num"], df["energy_error"], c=colors, s=60, zorder=3)
ax.set_yscale("log")
ax.axhline(y=CHEMICAL_ACCURACY_HA, color="gray", linestyle="--", linewidth=1,
           label=f"Chemical accuracy ({CHEMICAL_ACCURACY_HA} Ha)")

ax.set_xlabel("Experiment #", fontsize=12)
ax.set_ylabel("Energy Error (Ha, log scale)", fontsize=12)
ax.set_title("autoresearch-qc: VQE Ansatz Discovery", fontsize=14)
ax.legend(fontsize=10)

fig.tight_layout()
fig.savefig("images/progress.png", dpi=150)
plt.show()
print("Saved images/progress.png")

## 2. Circuit Complexity Over Time

Tracks whether the agent trends toward simpler or more complex circuits.

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 5))

ax1.set_xlabel("Experiment #", fontsize=12)
ax1.set_ylabel("n_params", color="#3498db", fontsize=12)
ax1.plot(df["experiment_num"], df["n_params"], "o-", color="#3498db", label="n_params")
ax1.tick_params(axis="y", labelcolor="#3498db")

ax2 = ax1.twinx()
ax2.set_ylabel("circuit_depth", color="#e67e22", fontsize=12)
ax2.plot(df["experiment_num"], df["circuit_depth"], "s-", color="#e67e22", label="circuit_depth")
ax2.tick_params(axis="y", labelcolor="#e67e22")

ax1.set_title("Circuit Complexity Over Time", fontsize=14)
fig.tight_layout()
plt.show()

## 3. Energy Convergence (Top 5 Experiments)

Compares wall time across the best-performing experiments.

In [ ]:
# Top 5 experiments by energy_error
top5 = df.nsmallest(5, "energy_error")

fig, ax = plt.subplots(figsize=(10, 5))
for _, row in top5.iterrows():
    label = f"#{int(row['experiment_num'])}: {row['energy_error']:.6f} Ha"
    ax.barh(label, row["wall_time"], color="#3498db", alpha=0.7)

ax.set_xlabel("Wall Time (s)", fontsize=12)
ax.set_title("Top 5 Experiments — Wall Time Comparison", fontsize=14)
fig.tight_layout()
plt.show()

print("\nTop 5 experiments:")
print(top5[["experiment_num", "energy_error", "energy_error_mha", "chemical_accuracy",
            "n_params", "circuit_depth", "wall_time", "notes"]].to_string(index=False))

## 4. Summary Statistics

In [ ]:
total = len(df)
best_idx = df["energy_error"].idxmin()
best = df.loc[best_idx]
chem_acc_achieved = df["chemical_accuracy"].any()

# Count improvements
improved = 0
best_so_far = float("inf")
for _, row in df.iterrows():
    if row["energy_error"] < best_so_far:
        improved += 1
        best_so_far = row["energy_error"]

print(f"Total experiments:         {total}")
print(f"Improvements kept:         {improved} ({improved/total*100:.0f}%)")
print(f"Best energy_error:         {best['energy_error']:.8f} Ha ({best['energy_error']*1000:.4f} mHa)")
print(f"Best n_params:             {int(best['n_params'])}")
print(f"Best circuit_depth:        {int(best['circuit_depth'])}")
print(f"Chemical accuracy reached: {chem_acc_achieved}")
if chem_acc_achieved:
    first_acc = df[df["chemical_accuracy"]].iloc[0]
    print(f"  First achieved at experiment #{int(first_acc['experiment_num'])}")